# Section 3: Robustness, Where You Go Beyond (28 pts)

ECON 5200 Midterm Project · Replication Study

**AI is allowed in this notebook**, as a documented co-pilot under P.R.I.M.E. File every exchange, in full, in [`ai-appendix.md`](../ai-appendix.md) under the exercise it belongs to. If you cannot explain a line, you cannot submit it.

At least four exercises the original did not run. Choose from modern estimators, a specification curve, sensitivity to unobserved confounding, inference (clustering at the level of treatment assignment, a wild cluster bootstrap if clusters are few), bandwidth or instrument strength, and placebo tests, or justify an alternative. For each one: what you did, what you found, and what it implies about the original claim.

## Setup

In [4]:
import sys
from pathlib import Path
from rdrobust import rdrobust

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

# The repository root, whether this runs from notebooks/ (Jupyter) or from the root (reproduce.py).
ROOT = Path.cwd() if (Path.cwd() / "download_data.py").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
(ROOT / "output").mkdir(exist_ok=True)

from download_data import MY_PAPER, PAPERS, fetch

## Data

In [6]:
# Set MY_PAPER once, in download_data.py. fetch() downloads into data/ only if a file is missing.
if MY_PAPER:
    paths = fetch(MY_PAPER)
    print(PAPERS[MY_PAPER]["paper"])
    # TODO: read the file(s), e.g. df = pd.read_stata(paths[0])
    dwi = pd.read_stata(paths[0])
    print(dwi.shape)
else:
    print("TODO: set MY_PAPER in download_data.py (castle, card, lee, dw, hansen or abortion).")

  data/hansen_dwi.dta  9.5 MB
Hansen (2015), punishment and deterrence in drunk driving
(214558, 12)


In [7]:
CUTOFF = 0.08
dwi["bac_min"] = dwi["low_score"].astype(float) / 1000
RUNNING = "bac_min"
H = 0.05
CONTROLS = ["male", "white", "aged", "acc"]

In [8]:
def rd(cutoff, h=None, kernel="triangular", controls=None, data=None, outcome="recidivism", donut=0.0):
    data = dwi if data is None else data
    d = data[(data[RUNNING] - cutoff).abs() >= donut]
    covs = d[controls] if controls else None
    kw = {"h": h} if h else {}
    r = rdrobust(y=d[outcome], x=d[RUNNING], c=cutoff, p=1, kernel=kernel, covs=covs, **kw)
    return {"jump": r.coef.iloc[0, 0], "se": r.se.iloc[0, 0], "p_robust": r.pv.iloc[2, 0],
            "h": r.bws.iloc[0, 0], "n": int(r.N_h[0] + r.N_h[1])}


def show(label, r):
    print(f"{label:<34} {r['jump']:+.4f} (SE {r['se']:.4f}, robust p {r['p_robust']:.3f}), h = {r['h']:.3f}, n = {r['n']:,}")

## The four exercises

### Exercise 1: New RD method at both cutoffs

**What I did.** I use rdrobust, a newer RD method that picks the bandwidth from the data and gives robust p-values. I run it at 0.08 and 0.15.

In [10]:
r1 = rd(0.08, controls=CONTROLS)
show("cutoff 0.08", r1)

r2 = rd(0.15, controls=CONTROLS)
show("cutoff 0.15", r2)

Mass points detected in the running variable.
cutoff 0.08                        -0.0173 (SE 0.0060, robust p 0.024), h = 0.027, n = 53,565
Mass points detected in the running variable.
cutoff 0.15                        -0.0075 (SE 0.0048, robust p 0.203), h = 0.029, n = 84,225


**What I found.** At 0.08 the jump is -0.017 where p= 0.024. At 0.15 it is -0.0075 where p=0.203.

**What it implies about the original claim.** The effect at 0.08 still holds, but it is about 20% smaller than the paper. The effect at 0.15 is no longer significant.

### Exercise 2: Different bandwidths

**What I did.** I run the 0.08 model with half the same and double paper's bandwidth.

In [11]:
r_half = rd(0.08, h=0.025, kernel="uniform", controls=CONTROLS)
show("h = 0.025", r_half)

r_same = rd(0.08, h=0.05, kernel="uniform", controls=CONTROLS)
show("h = 0.05", r_same)

r_double = rd(0.08, h=0.10, kernel="uniform", controls=CONTROLS)
show("h = 0.10", r_double)

Mass points detected in the running variable.
h = 0.025                          -0.0196 (SE 0.0057, robust p 0.131), h = 0.025, n = 50,016
Mass points detected in the running variable.
h = 0.05                           -0.0222 (SE 0.0043, robust p 0.035), h = 0.050, n = 93,899
Mass points detected in the running variable.
h = 0.10                           -0.0207 (SE 0.0034, robust p 0.000), h = 0.100, n = 169,916


**What I found.** The jump is -0.020, -0.022 and -0.021. The robust p-value is 0.13, 0.035 and 0.000.

**What it implies about the original claim.** The size of the effect stay the same which support the paper. But with the narrow bandwidth it is not significant, so the result is less precise than the paper shows. The 0.10 window also crosse 0.15, so that number is not clean.

### Exercise 3: Placebo cutoffs

**What I did.** I test fake cutoffs at 0.05 and 0.11, where the law does nothing. I use h = 0.025 so the windows don't reach the real cutoffs.

In [12]:
r_005 = rd(0.05, h=0.025, kernel="uniform", controls=CONTROLS)
show("placebo 0.05", r_005)

r_011 = rd(0.11, h=0.025, kernel="uniform", controls=CONTROLS)
show("placebo 0.11", r_011)

Mass points detected in the running variable.
placebo 0.05                       -0.0201 (SE 0.0129, robust p 0.131), h = 0.025, n = 16,417
Mass points detected in the running variable.
placebo 0.11                       +0.0005 (SE 0.0044, robust p 0.669), h = 0.025, n = 75,303


**What I found.** At 0.11 the jump is 0.0005 where p = 0.67. At 0.05 it is -0.020 where p = 0.13.

**What it implies about the original claim.** The 0.11 test is 0, as it should be. The 0.05 test is not significant, but it is as big as the real effect. It is noisy because few drivers have such low BAC, but part of the 0.08 jump may come from the data's shape, not only punishment.

### Exercise 4: Early vs late years

**What I did.** I split the data into 1999-2003 and 2004-2007 and run the 0.08 model in each part.

In [13]:
early = dwi[dwi["year"] <= 2003]
late = dwi[dwi["year"] >= 2004]

r_early = rd(0.08, h=0.05, kernel="uniform", controls=CONTROLS, data=early)
show("1999-2003", r_early)

r_late = rd(0.08, h=0.05, kernel="uniform", controls=CONTROLS, data=late)
show("2004-2007", r_late)

Mass points detected in the running variable.
1999-2003                          -0.0261 (SE 0.0063, robust p 0.224), h = 0.050, n = 48,065
Mass points detected in the running variable.
2004-2007                          -0.0182 (SE 0.0060, robust p 0.071), h = 0.050, n = 45,834


**What I found.** The jump is -0.026 in 1999-2003 where p = 0.22 and -0.018 in 2004-2007 where p = 0.07.

**What it implies about the original claim.** The effect goes the same way in both periods, so it is not from just a few years. But neither half is significant alone, so the paper needs all years together to be precise.